In [1]:
import sys
sys.path.insert(0, '..')

import json
import subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## Notebook 17: Signal Infrastructure

Notebook 16 produces signals and writes them to `signals_output.json`. That works fine when run manually. But a live trading system needs signals to be generated automatically, every morning before the market opens, without anyone touching Jupyter.

This notebook is about the operational layer: how signals get generated on a schedule, where they go, and how to read the history that builds up over time.

### The pipeline

```
7:00am (cron / bot trigger)
    └── python run_signals.py
            ├── fetches today's prices
            ├── generates signals for all validated pairs
            ├── writes signals_output.json   (overwritten; always current)
            └── appends signals_history.jsonl (append-only; full record)

Trading bot
    └── reads signals_output.json → acts on buy_ticker / sell_ticker / exposure
```

### Why two files?

`signals_output.json` is what the bot reads. It only needs to know today's answer; overwriting is fine.

`signals_history.jsonl` is the audit trail. Every run appends one line. Over time it becomes a record of exactly when each pair entered and exited positions, what the z-scores were each day, and how long trades lasted. That record is useful for two things: debugging ("why did the bot trade on March 6?") and eventually evaluating whether the live signals match what the backtest predicted.

## 1. The Script

`run_signals.py` lives in the project root. It imports everything from the existing modules: `strategy.pairs_config`, `strategy.live`, `data.loader`, and produces the same output as notebook 16, but without any charts or tables. Just the data.

Run it manually from the terminal:
```bash
cd /Users/manu/Quant\ Project
python run_signals.py
```

Expected output:
```
[2026-03-06] Fetching prices...
[2026-03-06] Last price date: 2026-03-05. Generating signals...
[2026-03-06] Written to signals_output.json
[2026-03-06] Appended to signals_history.jsonl
[2026-03-06] Active: KO/PEP SHORT, HD/LOW SHORT
```

Adding a new validated pair is one line in `strategy/pairs_config.py`; the script picks it up automatically on the next run.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()

# Run from the project root so signals_output.json and signals_history.jsonl
# are always written there, regardless of where this notebook is opened from.
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'run_signals.py')],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print("ERRORS:")
    print(result.stderr)

[2026-03-09] Fetching prices...
[2026-03-09] Last price date: 2026-03-06. Generating signals...
[2026-03-09] Written to signals_output.json
[2026-03-09] Appended to signals_history.jsonl
[2026-03-09] Active: KO/PEP SHORT, HD/LOW SHORT
[2026-03-09] HEALTH WARNING — KO/PEP: WARN_BOTH (pvalue=0.7658, hedge_ratio=-0.1007) — new entries blocked
[2026-03-09] HEALTH WARNING — NUE/STLD: WARN_PVALUE (pvalue=0.4040, hedge_ratio=0.6205) — new entries blocked
[2026-03-09] HEALTH WARNING — V/MA: WARN_PVALUE (pvalue=0.2786, hedge_ratio=0.6859) — new entries blocked
[2026-03-09] HEALTH WARNING — GS/MS: WARN_PVALUE (pvalue=0.1591, hedge_ratio=5.3780) — new entries blocked
[2026-03-09] HEALTH WARNING — HD/LOW: WARN_PVALUE (pvalue=0.8026, hedge_ratio=1.0880) — new entries blocked
[2026-03-09] HEALTH WARNING — TRV/CB: WARN_PVALUE (pvalue=0.5908, hedge_ratio=1.2031) — new entries blocked



## 2. Scheduling

### Mac / Linux: cron

Open the cron editor:
```bash
crontab -e
```

Add this line (runs at 7:00am Monday–Friday):
```
0 7 * * 1-5 cd /Users/manu/Quant\ Project && python3 run_signals.py >> logs/signals.log 2>&1
```

Create the log directory first:
```bash
mkdir -p /Users/manu/Quant\ Project/logs
```

### From a trading bot (Python)

If your bot is a Python process, it can trigger the script directly:
```python
import subprocess, json, sys

subprocess.run([sys.executable, 'run_signals.py'], cwd='/Users/manu/Quant Project')

with open('/Users/manu/Quant Project/signals_output.json') as f:
    signals = json.load(f)

for s in signals['signals']:
    if s['signal'] != 0:
        print(f"{s['pair']}: {s['signal_label']} : buy {s['buy_ticker']}, sell {s['sell_ticker']}")
        print(f"  Fixed exposure: ${s['fixed_dollar_exposure']:,.0f}")
```

### From any other bot

Shell out to the script however your bot's language supports it, then read `signals_output.json`. The file is always up to date after the script runs.

## 3. Signal History

`signals_history.jsonl` is a newline-delimited JSON file: one complete signal record per line, one line per run. As runs accumulate, it becomes a day-by-day log of every signal the strategy produced in live operation.

The cells below read and analyse that history. They work immediately after the first run and become more informative as more days accumulate.

In [3]:
history_path = PROJECT_ROOT / 'signals_history.jsonl'

if not history_path.exists() or history_path.stat().st_size == 0:
    print("No history yet — run the script at least once first.")
else:
    records = []
    with open(history_path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    rows = []
    for rec in records:
        for s in rec['signals']:
            rows.append({
                'date':                       rec['generated_at'],
                'last_price_date':            rec['last_price_date'],
                'pair':                       s['pair'],
                'signal':                     s['signal'],
                'signal_label':               s['signal_label'],
                'zscore':                     s['zscore'],
                'days_in_position':           s['days_in_position'],
                'fixed_dollar_exposure':      s['fixed_dollar_exposure'],
                'normalized_dollar_exposure': s['normalized_dollar_exposure'],
            })

    history_df = pd.DataFrame(rows)
    history_df['date'] = pd.to_datetime(history_df['date'])

    print(f"History: {len(records)} runs, {history_df['date'].nunique()} unique dates")
    print(f"Date range: {history_df['date'].min().date()} → {history_df['date'].max().date()}")
    print("\nMost recent run:")
    display(history_df[history_df['date'] == history_df['date'].max()]
            [['pair', 'signal_label', 'zscore', 'days_in_position', 'fixed_dollar_exposure']])

History: 2 runs, 1 unique dates
Date range: 2026-03-09 → 2026-03-09

Most recent run:


,pair,signal_label,zscore,days_in_position,fixed_dollar_exposure
0,KO/PEP,SHORT,0.814,19,-0.0
1,NUE/STLD,FLAT,-0.920,0,0.0
2,V/MA,FLAT,-0.111,0,0.0
3,GS/MS,FLAT,-0.932,0,0.0
4,HD/LOW,SHORT,0.177,6,-0.0
5,TRV/CB,FLAT,0.700,0,0.0
6,KO/PEP,SHORT,0.814,19,-0.0
7,NUE/STLD,FLAT,-0.920,0,0.0
8,V/MA,FLAT,-0.111,0,0.0
9,GS/MS,FLAT,-0.932,0,0.0


In [4]:
# Signal activity over time — how often was each pair active?
if 'history_df' in dir() and len(history_df) > 0:
    activity = (
        history_df[history_df['signal'] != 0]
        .groupby('pair')
        .agg(
            active_days=('signal', 'count'),
            long_days=('signal', lambda x: (x == 1).sum()),
            short_days=('signal', lambda x: (x == -1).sum()),
        )
        .sort_values('active_days', ascending=False)
    )
    total_runs = history_df['date'].nunique()
    activity['pct_active'] = (activity['active_days'] / total_runs).map('{:.1%}'.format)

    print(f"Signal activity across {total_runs} recorded run(s):")
    display(activity)
else:
    print("Not enough history yet — accumulate more runs first.")

Signal activity across 1 recorded run(s):


,active_days,long_days,short_days,pct_active
pair,,,,
HD/LOW,2,0,2,200.0%
KO/PEP,2,0,2,200.0%


In [5]:
# Signal changes — when did a pair enter or exit a position?
if 'history_df' in dir() and history_df['date'].nunique() > 1:
    changes = []
    for pair in history_df['pair'].unique():
        pair_df = history_df[history_df['pair'] == pair].sort_values('date')
        prev = pair_df['signal'].shift(1)
        changed = pair_df[pair_df['signal'] != prev].copy()
        changed['prev_signal'] = prev[changed.index]
        changed['event'] = changed.apply(
            lambda r: 'ENTRY' if r['signal'] != 0 else 'EXIT', axis=1
        )
        changes.append(changed[['date', 'pair', 'prev_signal', 'signal', 'signal_label', 'event', 'zscore']])

    changes_df = pd.concat(changes).sort_values('date')
    changes_df = changes_df[changes_df['prev_signal'].notna()]  # drop first row per pair
    print("Signal changes (entries and exits):")
    display(changes_df)
else:
    print("Need at least 2 runs to detect signal changes.")

Need at least 2 runs to detect signal changes.


> **Observations: Signal Infrastructure**
>
> The signal system works as designed. A single `python run_signals.py` invocation fetches prices, computes z-scores across all six pairs, runs health checks, writes `signals_output.json`, and appends a timestamped entry to `signals_history.jsonl`. The output is machine-readable JSON: pair, signal label, z-score, days in position, p-value, health status, hedge ratio, and dollar exposures. Everything needed to act on the signal is in one file.
>
> The health check layer catches cointegration breakdown in real time. On the current run, five of six pairs show `WARN_PVALUE` (p-value above the stable threshold), meaning recent price action has temporarily weakened the statistical relationship. This is expected and exactly what the system is designed to flag: health warnings suppress new entries while allowing existing positions to continue and exit normally. A warning does not mean the pair is broken; it means the system is being appropriately conservative until the relationship restores.
>
> The audit log in `signals_history.jsonl` is append-only. Every run is recorded with its timestamp, p-values, z-scores, and signals. This makes it possible to reconstruct exactly what the system saw on any given day, which is essential for post-trade analysis and for confirming that no forward-looking data entered any live decision.

## 4. What Was Built

- **`run_signals.py`:** standalone script in the project root. Fetches prices, generates signals for all validated pairs in `strategy/pairs_config.py`, writes `signals_output.json` (overwrite) and appends to `signals_history.jsonl`. Runnable from cron, a bot subprocess call, or the terminal. Adding a new pair requires no changes to this script.
- **`signals_history.jsonl`:** append-only log. One JSON line per run. Accumulates a day-by-day record of every signal, z-score, and position size the strategy produced in live operation.
- **Notebook 17:** explains the pipeline, shows how to schedule the script, and provides history analysis cells that grow more useful as runs accumulate.